In [1]:
from pymongo import MongoClient
import pandas as pd

In [4]:
mongo_uri = "mongodb://localhost:27017/"
db_name = "FinancialFraud"
collection_name = 'transactions'

In [5]:
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

In [6]:
def extract_data(path):
    df = pd.read_csv(path)
    return df

In [7]:
def transform_data(df):
    df = df.drop_duplicates()
    df = df.dropna()
    df['step'] = df['step'].astype(int)
    df['amount'] = df['amount'].astype(float)
    df['isFraud'] = df['isFraud'].astype(int)
    df['isFlaggedFraud'] = df['isFlaggedFraud'].astype(int)

    df['RiskLevel'] = df['amount'].apply(lambda x : 'High' if x>1000 else 'Low')

    df["balanceDiffOrig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balanceDiffDest"] = df["newbalanceDest"] - df["oldbalanceDest"]

    return df.to_dict(orient='records')

In [8]:
def load_date(records):
    if records:
        collection.insert_many(records)
        print(f'Inserted {len(records)} documents')

In [9]:
if __name__ == "__main__":
    path = '/Users/alfiyaansari/Desktop/Project/PS_20174392719_1491204439457_log.csv'
    df = extract_data(path)
    records = transform_data(df)
    load_date(records)

Inserted 6362620 documents


In [11]:
#Create Indexes
collection.create_index([("isFraud", 1)])
collection.create_index([("amount", -1)])
collection.create_index([('step', 1)])
collection.create_index([('type',1)])

'type_1'